# Распределения на квене

Это тестовый файлик, пока играемся

In [1]:
import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM
from typing import Dict, List, Any

MODEL_NAME = "Qwen/Qwen3-4B-Base"
INITIAL_PROMPT = "Искусственный интеллект меняет "
TOP_K = 3
MAX_DEPTH = 4
OUTPUT_FILE = "token_tree.json"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True
)
model.eval()

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (post_attention_layer

In [2]:
def build_prob_tree(
    model,
    tokenizer,
    input_ids: torch.Tensor,
    prompt_str: str,
    top_k: int = 3,
    max_depth: int = 4,
    depth: int = 0
) -> Dict[str, Any]:
    
    with torch.no_grad():
        outputs = model(input_ids)
        logits = outputs.logits[:, -1, :]
        probs = torch.softmax(logits, dim=-1)
        top_probs, top_indices = torch.topk(probs, top_k)

    node = {
        "prompt_so_far": prompt_str,
        "depth": depth,
        "candidates": []
    }

    for prob, idx in zip(top_probs[0].tolist(), top_indices[0].tolist()):
        token_str = tokenizer.decode([idx], skip_special_tokens=True)
        
        is_space = (
            isinstance(token_str, str) and 
            (token_str.strip() == "" or any(c.isspace() for c in token_str))
        )

        child = {
            "token": token_str,
            "probability": round(prob, 6),
            "is_space_token": is_space,
            "children": []
        }

        if not is_space and depth < max_depth:
            new_token = torch.tensor([[idx]], device=model.device)
            new_input_ids = torch.cat([input_ids, new_token], dim=1)
            child["children"] = build_prob_tree(
                model, tokenizer, new_input_ids, prompt_str + token_str,
                top_k, max_depth, depth + 1
            )

        node["candidates"].append(child)

    return node


In [3]:
inputs = tokenizer(INITIAL_PROMPT, return_tensors="pt")
input_ids = inputs.input_ids.to(model.device)

tree = build_prob_tree(model, tokenizer, input_ids, INITIAL_PROMPT, TOP_K, MAX_DEPTH)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(tree, f, ensure_ascii=False, indent=2)
print(f" Дерево сохранено в {OUTPUT_FILE}")



 Дерево сохранено в token_tree.json


In [4]:
def print_tree_readable(node, indent=0):
    """Человекочитаемый вывод дерева"""
    prefix = "  " * indent
    
    if indent == 0:
        print(f"\n📝 Промпт: \"{node.get('prompt_so_far', '')}\"\n")
    
    # Безопасно получаем список кандидатов
    candidates = node.get("candidates", [])
    if not candidates:
        return

    for i, child in enumerate(candidates, 1):
        token = repr(child["token"])
        prob = child["probability"]
        flag = " ⛔(SPACE)" if child.get("is_space_token") else ""
        
        if indent == 0:
            print(f"{i}. {token} | p={prob:.4f}{flag}")
        else:
            print(f"{prefix}└─ {token} | p={prob:.4f}{flag}")
        
        if child.get("children"):
            print_tree_readable(child["children"], indent + 1)

with open("token_tree.json", "r", encoding="utf-8") as f:
    tree = json.load(f)
    
print_tree_readable(tree)


📝 Промпт: "Искусственный интеллект меняет "

1. '2' | p=0.2130
  └─ '0' | p=0.6411
    └─ '2' | p=0.6035
      └─ '1' | p=0.2627
        └─ ' год' | p=0.7012 ⛔(SPACE)
        └─ '-' | p=0.1199
        └─ '\n' | p=0.0222 ⛔(SPACE)
      └─ '2' | p=0.2356
        └─ ' год' | p=0.6294 ⛔(SPACE)
        └─ '-' | p=0.0996
        └─ '\n' | p=0.0402 ⛔(SPACE)
      └─ '0' | p=0.2319
        └─ ' год' | p=0.4390 ⛔(SPACE)
        └─ '-' | p=0.2791
        └─ '\n' | p=0.0384 ⛔(SPACE)
    └─ '1' | p=0.2151
      └─ '9' | p=0.3406
        └─ ' год' | p=0.6069 ⛔(SPACE)
        └─ '-' | p=0.1609
        └─ '\n' | p=0.0312 ⛔(SPACE)
      └─ '8' | p=0.2491
        └─ ' год' | p=0.5977 ⛔(SPACE)
        └─ '-' | p=0.1465
        └─ '\n' | p=0.0359 ⛔(SPACE)
      └─ '7' | p=0.1659
        └─ ' год' | p=0.5791 ⛔(SPACE)
        └─ '-' | p=0.1355
        └─ '\n' | p=0.0475 ⛔(SPACE)
    └─ '0' | p=0.0257
      └─ '0' | p=0.2690
        └─ '-' | p=0.2023
        └─ '0' | p=0.1504
        └─ ' лет' | p=0.1050 ⛔

In [8]:
def print_tree_readable(node, indent=0):
    """Человекочитаемый вывод дерева"""
    prefix = "  " * indent
    
    if indent == 0:
        print(f"\n📝 Промпт: \"{node.get('prompt_so_far', '')}\"\n")
    
    # Безопасно получаем список кандидатов
    candidates = node.get("candidates", [])
    if not candidates:
        return

    for i, child in enumerate(candidates, 1):
        token = child["token"]
        prob = child["probability"]
        flag = " ⛔(SPACE)" if child.get("is_space_token") else ""
        
        if indent == 0:
            print(f"{i}. {node.get('prompt_so_far', '')}{token} | p={prob:.4f}{flag}")
        else:
            print(f"{prefix}└─ {node.get('prompt_so_far', '')}{token} | p={prob:.4f}{flag}")
        
        if child.get("children"):
            print_tree_readable(child["children"], indent + 1)

with open("token_tree.json", "r", encoding="utf-8") as f:
    tree = json.load(f)
    
print_tree_readable(tree)


📝 Промпт: "Искусственный интеллект меняет "

1. Искусственный интеллект меняет 2 | p=0.2130
  └─ Искусственный интеллект меняет 20 | p=0.6411
    └─ Искусственный интеллект меняет 202 | p=0.6035
      └─ Искусственный интеллект меняет 2021 | p=0.2627
        └─ Искусственный интеллект меняет 2021 год | p=0.7012 ⛔(SPACE)
        └─ Искусственный интеллект меняет 2021- | p=0.1199
        └─ Искусственный интеллект меняет 2021
 | p=0.0222 ⛔(SPACE)
      └─ Искусственный интеллект меняет 2022 | p=0.2356
        └─ Искусственный интеллект меняет 2022 год | p=0.6294 ⛔(SPACE)
        └─ Искусственный интеллект меняет 2022- | p=0.0996
        └─ Искусственный интеллект меняет 2022
 | p=0.0402 ⛔(SPACE)
      └─ Искусственный интеллект меняет 2020 | p=0.2319
        └─ Искусственный интеллект меняет 2020 год | p=0.4390 ⛔(SPACE)
        └─ Искусственный интеллект меняет 2020- | p=0.2791
        └─ Искусственный интеллект меняет 2020
 | p=0.0384 ⛔(SPACE)
    └─ Искусственный интеллект меняет 201 